# Email A/B Test: Did the Emails Work?

In [6]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, "../src")
from ab_tools import bootstrap_diff, holm_adjust, sample_size_per_group, srm_check, two_proportion_test

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a" # control, mens email, womens email
df = pd.read_csv("../data/hillstrom.csv")
print(df.shape)
df.head()

(64000, 12)


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


In [7]:
print("Missing values:", df.isna().sum().sum())
print(df["segment"].value_counts())
df.describe().round(2)

Missing values: 0
segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64


,recency,history,mens,womens,newbie,visit,conversion,spend
count,64000.00,64000.00,64000.00,64000.00,64000.0,64000.00,64000.00,64000.00
mean,5.76,242.09,0.55,0.55,0.5,0.15,0.01,1.05
std,3.51,256.16,0.50,0.50,0.5,0.35,0.09,15.04
min,1.00,29.99,0.00,0.00,0.0,0.00,0.00,0.00
25%,2.00,64.66,0.00,0.00,0.0,0.00,0.00,0.00
50%,6.00,158.11,1.00,1.00,1.0,0.00,0.00,0.00
75%,9.00,325.66,1.00,1.00,1.0,0.00,0.00,0.00
max,12.00,3345.93,1.00,1.00,1.0,1.00,1.00,499.00


In [8]:
counts = df["segment"].value_counts()
p_srm = srm_check(counts.tolist())
print(f"Sample ratio mismatch p-value: {p_srm:.3f}")
print("RED FLAG" if p_srm < 0.01 else "Split looks fine: about one-third each")

Sample ratio mismatch p-value: 0.904
Split looks fine: about one-third each


In [9]:
# Randomization check: customer features should look the same in every group
df.groupby("segment")[["recency", "history", "mens", "womens", "newbie"]].mean().round(3)

,recency,history,mens,womens,newbie
segment,,,,,
Mens E-Mail,5.774,242.836,0.551,0.551,0.502
No E-Mail,5.750,240.883,0.553,0.548,0.502
Womens E-Mail,5.768,242.537,0.549,0.550,0.503


In [10]:
for metric, baseline in [("visit", 0.106), ("conversion", 0.0057)]:
    n = sample_size_per_group(baseline, 0.01 if metric == "visit" else 0.003)
    print(f"{metric}: need {n:,} per group")

visit: need 15,487 per group
conversion: need 12,464 per group


## Data and experiment checks
The data has 64,000 customers and no missing values. The three groups are nearly equal in size (SRM p = 0.904) and look alike on recency, past spend, and product history, so randomization worked. Each group has about 21,300 customers, more than the 15,487 needed to detect a 1-point lift in visits.